# T02 — So sánh PhoBERT A với mô hình truyền thống

Notebook chỉ đánh giá các artifact đã đóng băng trên UIT-ViSFD test. Không train, không chọn siêu tham số từ test. Seed cố định: `42`.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/firefly-cinema')]
PROJECT_ROOT = next(path for path in candidates if (path / 'ml' / 'src' / 'eval_utils.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'ml' / 'src'))
from eval_utils import RANDOM_STATE, load_test, normalize, run_comparison
print('PROJECT_ROOT =', PROJECT_ROOT)
print('RANDOM_STATE =', RANDOM_STATE)

## Gate dữ liệu và tiền xử lý

So sánh trực tiếp 20 câu đầu giữa `text_norm` đã đóng băng và `normalize(text_raw)` theo contract backend hiện hành.

In [ ]:
test = load_test(PROJECT_ROOT / 'data' / 'uit-visfd' / 'processed' / 'test.csv')
assert len(test) == 2224
assert test['has_negative'].value_counts().sort_index().to_dict() == {0: 981, 1: 1243}
test[['text_raw', 'text_norm', 'normalize']].head(20)

## Đánh giá và benchmark CPU

Cả bốn mô hình dùng `compute_metrics`. CPU benchmark dùng batch 1, warm-up 1 câu, 1.000 câu đầu và `time.perf_counter`. Giá trị GPU PhoBERT chỉ là tham khảo từ runtime lịch sử, không so trực tiếp với benchmark CPU batch 1.

In [ ]:
comparison, environment = run_comparison(PROJECT_ROOT, benchmark_count=1000)
display(comparison)
environment

## Acceptance checks và blocker tiền xử lý

In [ ]:
required = {'model','input','precision_neg','recall_neg','f1_neg','macro_f1','accuracy','ms_cpu','ms_gpu','size_mb'}
assert required == set(comparison.columns)
assert len(comparison) >= 8 and comparison['model'].nunique() >= 4
phobert_f1 = comparison.query("model == 'PhoBERT A' and input == 'text_norm'")['f1_neg'].iloc[0]
assert abs(phobert_f1 - 0.918) <= 0.005
pivot = comparison.pivot(index='model', columns='input', values='f1_neg')
pivot['delta_abs'] = (pivot['text_norm'] - pivot['normalize']).abs()
display(pivot)
blockers = pivot[pivot['delta_abs'] > 0.01]
print('BLOCKER preprocessing (> 0.01):' if len(blockers) else 'Không có blocker preprocessing > 0.01')
display(blockers)